In [1]:
import firebase_admin
from firebase_admin import credentials, firestore, storage, db
import pandas as pd
import os 
import dotenv
dotenv.load_dotenv()

True

# Firebase Init

In [2]:
service_account_info = {
    "type": os.getenv("FIREBASE_TYPE"),
    "project_id": os.getenv("FIREBASE_PROJECT_ID"),
    "project_key_id": os.getenv("FIREBASE_PRIVATE_KEY_ID"),
    "private_key": os.getenv("FIREBASE_PRIVATE_KEY"),
    "client_email": os.getenv("FIREBASE_CLIENT_EMAIL"),
    "client_id": os.getenv("FIREBASE_CLIENT_ID"),
    "auth_uri": os.getenv("FIREBASE_AUTH_URI"),
    "token_uri": os.getenv("FIREBASE_TOKEN_URI"),
    "auth_provider_x509_cert_url": os.getenv("FIREBASE_AUTH_PROVIDER_X509_CERT_URL"),
    "client_x509_cert_url": os.getenv("FIREBASE_CLIENT_X509_CERT_URL"),
    "universe_domain": os.getenv("FIREBASE_UNIVERSE_DOMAIN"),
}

In [ ]:
#Add your own data for storageBucket and databaseURL
cred = credentials.Certificate(service_account_info)
firebase_admin.initialize_app(cred, {
    'storageBucket': '',
    'databaseURL': ''
})

In [4]:
#Used to store the images
bucket = storage.bucket()

## Upload Data

In [5]:
image_folder_path = './products/images/'

In [10]:
product_collection = db.reference('products')

In [11]:
df = pd.read_json('./products/products.jsonl',orient = 'records', lines = True)
df.head(2)

,name,category,description,ingredients,sizes,price,rating,calories,image_path,syrup
0,Cappuccino,Coffee,A rich and creamy cappuccino made with freshly...,"[Espresso, Steamed Milk, Milk Foam]","{'small': {'price': 3.75, 'calories': 80}, 'me...",4.50,4.7,110.0,cappuccino.jpg,True
1,Jumbo Savory Scone,Bakery,"Deliciously flaky and buttery, this jumbo savo...","[Flour, Butter, Cheese, Herbs, Baking Powder, ...",NaN,3.25,4.3,400.0,SavoryScone.webp,False


In [12]:
def upload_image(bucket, image_path):
    image_name = image_path.split('/')[-1]
    blob = bucket.blob(f'products_images/{image_name}')

    #Upload image
    blob.upload_from_filename(image_path)
    
    #Make the image publicly accessible
    blob.make_public()

    return blob.public_url

In [14]:
for _, row in df.iterrows():
    image_path = os.path.join(image_folder_path, row["image_path"])
    image_url = upload_image(bucket, image_path) 

    raw = row.to_dict()

    for optional_key in ["sizes"]:
        if optional_key in raw and (pd.isna(raw[optional_key]) or raw[optional_key] is None):
            raw.pop(optional_key, None)

    raw.pop("image_path", None)  
    raw["image_url"] = image_url

    product_collection.push().set(raw)  


UnknownError: Unknown error while making a remote service call: Out of range float values are not JSON compliant